In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ============================================================
# 1. Load data
# ============================================================

df = pd.read_csv("dataset.csv")



ModuleNotFoundError: No module named 'sklearn'

In [ ]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

CSV_PATH = "dataset.csv"
TARGET = "orders_total"
DATE_COLUMN = "date"

TEST_DAYS = 10

df = pd.read_csv(CSV_PATH)
df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN])
df = df.sort_values(DATE_COLUMN).reset_index(drop=True)

print("=" * 70)
print("DATASET")
print("=" * 70)
print(f"Rows: {len(df)}")
print(f"Start: {df[DATE_COLUMN].min().date()}")
print(f"End:   {df[DATE_COLUMN].max().date()}")

for lag in [1, 2, 3, 7, 14]:
    df[f"orders_lag_{lag}"] = df[TARGET].shift(lag)

df["orders_roll_7"] = df[TARGET].shift(1).rolling(window=7).mean()
df["orders_roll_14"] = df[TARGET].shift(1).rolling(window=14).mean()

df["day_of_week"] = df[DATE_COLUMN].dt.dayofweek
df["is_weekend"] = (df["day_of_week"] >= 4).astype(int)

weekday_dummies = pd.get_dummies(
    df["day_of_week"],
    prefix="weekDay",
    dtype=float
)

for i in range(7):
    column = f"weekDay_{i}"
    if column not in weekday_dummies.columns:
        weekday_dummies[column] = 0.0

weekday_dummies = weekday_dummies[
    [f"weekDay_{i}" for i in range(7)]
]

df = pd.concat([df, weekday_dummies], axis=1)

df["trend"] = np.arange(len(df))

features_to_delete = [
    'meta_leads',
    'meta_cpc_usd',
    'meta_cpm_usd',
    'meta_active_campaigns',
    # 'orders_with_discount',
    'meta_results_reported',
    'google_ads_cost_usd_estimated',
    'google_ads_avg_cpc_usd',
    'blogger_spend_afn',
    'total_ad_spend_afn',
    'cancelled_orders',
    'successful_orders',
    'kabul_order_count',
    'herat_order_count',
    'online_payment_orders',
    'average_order_item_count',
    'average_order_value',
    'first_orders_count',
    'repeat_orders_count',
    'delivery_share_lag7_avg',
    'delivery_share_lag1',
    'avg_delivery_fee_lag7_avg',
    'avg_delivery_fee_lag1',
    'avg_discount_amount_lag1',
    'avg_discount_amount_lag7_avg',
    'new_active_restaurants',
    'rating_sum_that_day',
    'featured_that_day',
    'commission_sum_that_day',
    'new_available_dishes',
    'count_that_day',
    'event_bookings',
    'new_featured_dishes',
    'cumulative_featured_restaurants',
    'cumulative_commission_sum',
    'avg_commission_rate',
    'blogger_views_reach_lag7_avg',
    'blogger_views_reach_lag1',
    'spend_lag_1',
    'meta_impressions_lag1',
    'meta_impressions_lag7_avg',
    'meta_link_clicks_lag1',
    'meta_ctr_percent_lag7_avg',
    'meta_cpm_usd_lag1',
    'meta_cpc_usd_lag7_avg',
    'google_ads_ctr_percent_estimated',
    'google_ads_ctr_percent_estimated_lag1',
    'google_ads_ctr_percent_estimated_lag7_avg',
    'avg_item_price_lag1',
    'avg_item_price_lag7_avg',
    'avg_order_food_value_lag1',
    'avg_order_food_value_lag7_avg',
    'day_of_week_num'
]

features_to_delete = list(set(features_to_delete))
features_to_delete = [
    col for col in features_to_delete
    if col in df.columns
]

df = df.drop(columns=features_to_delete, errors="ignore")

print(f"\nDeleted {len(features_to_delete)} columns")

numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()

for col in [
    TARGET,
    "day_of_week",
    "day_of_week_num",
    "Unnamed: 0"
]:
    if col in numeric_columns:
        numeric_columns.remove(col)

FEATURES = numeric_columns

print("\n" + "=" * 70)
print("BASE FEATURES")
print("=" * 70)
print(f"Total features: {len(FEATURES)}")

for i, feature in enumerate(FEATURES, 1):
    print(f"{i}. {feature}")

test_start = len(df) - TEST_DAYS

train = df.iloc[:test_start].copy()
test = df.iloc[test_start:].copy()

print("\n" + "=" * 70)
print("TRAIN / TEST SPLIT")
print("=" * 70)
print(
    f"Training period: {train[DATE_COLUMN].min().date()} → "
    f"{train[DATE_COLUMN].max().date()}"
)
print(
    f"Testing period:  {test[DATE_COLUMN].min().date()} → "
    f"{test[DATE_COLUMN].max().date()}"
)
print(f"Training samples: {len(train)}")
print(f"Testing samples:  {len(test)}")

X_train = train[FEATURES].copy()
y_train = train[TARGET].copy()

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("linear_regression", LinearRegression())
])

model.fit(X_train, y_train)

print("\nModel trained.")

history = {
    i: df.loc[i, TARGET]
    for i in range(test_start)
}

def build_future_features(row_index, history):
    values = {}

    current_row = df.loc[row_index]

    for feature in FEATURES:
        if feature.startswith("orders_"):
            continue

        values[feature] = current_row[feature]

    for lag in [1, 2, 3, 7, 14]:
        previous_index = row_index - lag
        values[f"orders_lag_{lag}"] = history.get(
            previous_index,
            np.nan
        )

    previous_values_7 = [
        history.get(i, np.nan)
        for i in range(row_index - 7, row_index)
    ]
    previous_values_7 = [
        v for v in previous_values_7
        if pd.notna(v)
    ]

    values["orders_roll_7"] = (
        np.mean(previous_values_7)
        if previous_values_7
        else np.nan
    )

    previous_values_14 = [
        history.get(i, np.nan)
        for i in range(row_index - 14, row_index)
    ]
    previous_values_14 = [
        v for v in previous_values_14
        if pd.notna(v)
    ]

    values["orders_roll_14"] = (
        np.mean(previous_values_14)
        if previous_values_14
        else np.nan
    )

    return pd.DataFrame(
        [values],
        columns=FEATURES
    )

predictions = []

for i in range(test_start, len(df)):
    X_future = build_future_features(i, history)

    prediction = model.predict(X_future)[0]
    prediction = max(0.0, prediction)

    predictions.append(prediction)
    history[i] = prediction

results = test[
    [DATE_COLUMN, TARGET]
].copy()

results["predicted_orders"] = predictions
results["error"] = (
    results["predicted_orders"] - results[TARGET]
)
results["absolute_error"] = results["error"].abs()

results["prediction_accuracy"] = np.where(
    results[TARGET] != 0,
    (
        1 -
        results["absolute_error"] /
        results[TARGET].abs()
    ) * 100,
    np.nan
)

results["prediction_accuracy"] = (
    results["prediction_accuracy"]
    .clip(lower=0, upper=100)
)

mae = mean_absolute_error(
    results[TARGET],
    results["predicted_orders"]
)

rmse = np.sqrt(
    mean_squared_error(
        results[TARGET],
        results["predicted_orders"]
    )
)

r2 = r2_score(
    results[TARGET],
    results["predicted_orders"]
)

non_zero_actual = results[TARGET] != 0

mape = (
    results.loc[
        non_zero_actual,
        "absolute_error"
    ]
    /
    results.loc[
        non_zero_actual,
        TARGET
    ].abs()
).mean() * 100

overall_accuracy = np.clip(
    100 - mape,
    0,
    100
)

print("\n" + "=" * 90)
print("DAILY PREDICTIONS")
print("=" * 90)

display_results = results[
    [
        DATE_COLUMN,
        TARGET,
        "predicted_orders",
        "error",
        "absolute_error",
        "prediction_accuracy"
    ]
].copy()

print(
    display_results.to_string(
        index=False,
        formatters={
            "predicted_orders": "{:.2f}".format,
            "error": "{:.2f}".format,
            "absolute_error": "{:.2f}".format,
            "prediction_accuracy": (
                lambda x:
                "N/A"
                if pd.isna(x)
                else f"{x:.2f}%"
            )
        }
    )
)

print("\n" + "=" * 90)
print("MODEL PERFORMANCE")
print("=" * 90)
print(f"Overall Accuracy : {overall_accuracy:.2f}%")
print(f"MAPE             : {mape:.2f}%")
print(f"MAE              : {mae:.2f} orders")
print(f"RMSE             : {rmse:.2f}")
print(f"R²               : {r2:.4f}")
print(
    f"\nAverage absolute error: "
    f"{results['absolute_error'].mean():.2f} orders"
)
print(
    f"Largest prediction error: "
    f"{results['absolute_error'].max():.2f} orders"
)